# MiniMax H3 — Kaggle T4×2 · Ultra-Q2 超小模型版
默认 GGUF Q2 + stable-diffusion.cpp，不加载 40–50 GB 的 ComfyUI INT8 方案。支持 T2V/I2V/R2V 各自模型，优先尝试双 T4 分层计算同一视频；在 Kaggle T4×2 上仍未经完整验收。
生成文件在 `/kaggle/working/h3_videos/`。模型/编译文件在 `/kaggle/temp/`，会随 Kaggle 会话销毁。保留旧的 ComfyUI Compact 入口供手动选择。


In [ ]:
import os, sys, subprocess
from pathlib import Path
ROOT=Path('/kaggle/working/shipinshencheng')
if (ROOT/'.git').exists():
    subprocess.run(['git','-C',str(ROOT),'pull','--ff-only'],check=True)
else:
    subprocess.run(['git','clone','--depth=1','https://github.com/mjw3177109/shipinshencheng.git',str(ROOT)],check=True)
print('GitHub project synced:', ROOT)


## 视频模式和硬盘档位
`BACKEND='ultra-q2'`：约 21GB 模型，无 Turbo LoRA，20 步，独立 C++ 推理后端；`BACKEND='comfy-compact'`：原约 42GB 的 ComfyUI 方案。Ultra 使用 6.26 GiB Q2 DiT、7.91 GiB Q2 文本编码器以及 VAE；会先预检实际下载量。
首次自动 T2V 短测试（384×224、22 帧）。I2V/R2V 需将原 JSON 指定的图片放到 `/kaggle/working/h3_inputs/`。原有模式严格区分，绝不自动转换 R2V↔I2V。


In [ ]:
import os, sys, subprocess
BACKEND='ultra-q2'  # 'ultra-q2' (smallest) / 'comfy-compact' (fallback)
MODE='t2v'  # 't2v', 'i2v', 'r2v'
WIDTH=384
HEIGHT=384 if MODE=='i2v' else 224
FRAMES=22
ULTRA_STEPS=20  # no Turbo LoRA required; slower, but smaller on disk
SINGLE_GPU=False  # False: experimental diffusion=cuda0&cuda1 layer splitting
os.environ.update({'H3_MODE':MODE, 'H3_WIDTH':str(WIDTH), 'H3_HEIGHT':str(HEIGHT),
                   'H3_FRAMES':str(FRAMES), 'H3_ULTRA_STEPS':str(ULTRA_STEPS),
                   'H3_SINGLE_GPU':'1' if SINGLE_GPU else '0'})
if BACKEND=='ultra-q2':
    subprocess.run([sys.executable,'-u',str(ROOT/'kaggle'/'ultra_gguf_h3.py')],check=True)
elif BACKEND=='comfy-compact':
    os.environ.update({'H3_STORAGE_PROFILE':'compact','H3_TURBO':'1'})
    subprocess.run([sys.executable,'-u',str(ROOT/'kaggle'/'bootstrap_h3.py')],check=True)
    subprocess.run([sys.executable,'-u',str(ROOT/'kaggle'/'run_from_sources.py')],check=True)
else:
    raise ValueError('Unknown BACKEND: '+BACKEND)


## 视频结果与验收报告
视频会保存在 `/kaggle/working/h3_videos/`；请从 Kaggle Output 下载。报告为 `render_report.json`，错误日志为 `comfyui.log`。当模型下载、CPU RAM 或显存不足时，不会生成虚假的 MP4。


In [ ]:
import json
from pathlib import Path
from IPython.display import Video,display
out=Path('/kaggle/working/h3_videos')
for name in ('ultra_report.json','render_report.json'):
    rp=out/name
    if rp.is_file(): print(name,rp.read_text(encoding='utf-8')[:6000])
videos=sorted(out.rglob('*.mp4'),key=lambda p:p.stat().st_mtime,reverse=True)
print('MP4 count:',len(videos))
if videos: display(Video(str(videos[0]),embed=True,width=640))
